In [29]:
import random
import pandas as pd

In [30]:
# Pour la suite, une instance de dégâts est appelée occurrence.
# Pour différencier les occurrences liées aux attaques de celles à liées à des effets, on rajoute la lettre A à côté de l'occurrence pour la différencier.

In [31]:
def _parse_occurrence(occ):
    if isinstance(occ, str) and occ.upper().endswith('A'):
        return int(occ[:-1]), True
    return int(occ), False

In [40]:
def _simulate_process(N, p, occurrences_parsed, trigger_soul, trigger_blank):
    """Simule une série d'occurrences de dégâts et renvoie X (total des dégâts effectifs)."""
    deck = ['Climax'] * p + ['Non-Climax'] * (N - p)
    random.shuffle(deck)
    discard = []
 
    trigger_deck = ['Soul trigger'] * trigger_soul + ['No soul trigger'] * trigger_blank
    random.shuffle(trigger_deck)
    trigger_discard = []
 
    clock_zone = []
    total_kept = 0
 
    for base_k, is_A in occurrences_parsed:
        k = base_k
 
        # --- résolution trigger pour occurrence A ---
        if is_A:
            if not trigger_deck and trigger_discard:
                trigger_deck = trigger_discard[:]
                trigger_discard = []
                random.shuffle(trigger_deck)
            if trigger_deck:
                card = trigger_deck.pop()
                trigger_discard.append(card)
                if card == 'Soul trigger':
                    k += 1
 
        # --- résolution du tirage principal de l'occurrence ---
        drawn_this_occurrence = []
        occurrence_failed = False
        halted = False
        draws_done = 0
 
        while draws_done < k:
            if len(deck) == 0:
                # système de refresh
                if not discard:
                    halted = True
                    break
                new_deck = discard[:]
                discard = []
                random.shuffle(new_deck)
                clock_card = new_deck.pop()  # carte au hasard -> clock zone
                clock_zone.append(clock_card)
                total_kept += 1 
                deck = new_deck
                if len(deck) == 1:
                    halted = True
                    break
                continue
 
            card = deck.pop()
            drawn_this_occurrence.append(card)
            draws_done += 1
            if card == 'Climax':
                occurrence_failed = True
                break
 
        if halted:
            discard.extend(drawn_this_occurrence)
            break
 
        if occurrence_failed:
            discard.extend(drawn_this_occurrence)
        else:
            # dans le cas où aucune climax n'est révélée, on incrémente le total des dégâts et on place toutes les cartes révélées dans la clock zone
            total_kept += len(drawn_this_occurrence)
            clock_zone.extend(drawn_this_occurrence)
 
            # système de level-up
            while len(clock_zone) >= 7:
                batch = clock_zone[:7]
                clock_zone = clock_zone[7:]
                white_idx = next((i for i, c in enumerate(batch) if c == 'Non-Climax'), None)
                if white_idx is not None:
                    batch.pop(white_idx)
                else:
                    batch.pop(0)
                discard.extend(batch)
 
    return total_kept

In [41]:
def simulation(N_list, p_list, occurrences, trigger_soul, trigger_blank, n_trials=100000):
    """
    Lance une simulation de type MonteCarlo avec les paramètres indiqués et renvoie un DataFrame correspondant
 
    Paramètres
    ----------
    N_list : list[int]        tailles de decks a tester
    p_list : list[int]        nombres de climax a tester
    occurrences : list        dégâts associés au finisher
    trigger_soul : int        nombre de cartes avec trigger dans le deck trigger
    trigger_blank : int       nombre de cartes sans trigger dans le deck trigger
    n_trials : int            nombre d'essais Monte Carlo par combinaison [N,p]
 
    Retour
    ------
    Chaque ligne du tableau correspond à une combinaison de deck size / nombre de climax parmi celles entrées
    Chaque colonne correspond à la probabilité jusqu'au moins X dégâts aient été faits au cours des occurrences
    """
    occurrences_parsed = [_parse_occurrence(o) for o in occurrences]
    max_possible = sum(k + (1 if is_A else 0) for k, is_A in occurrences_parsed)
 
    rows = []
    for N in N_list:
        for p in p_list:
            totals = [
                _simulate_process(N, p, occurrences_parsed, trigger_soul, trigger_blank)
                for _ in range(n_trials)
            ]
            row = {'N': N, 'p': p}
            for x in range(0, max_possible + 1):
                row[f'X_min={x}'] = round(sum(1 for t in totals if t >= x) / n_trials, 4)
            rows.append(row)
 
    return pd.DataFrame(rows)

In [42]:
table = simulation(
        N_list=[20,25,30],
        p_list=[4,6,8],
        occurrences=[4,'3A',4,'3A',4,'3A'],
        trigger_soul = 15,
        trigger_blank = 35,
        n_trials=10000,
        )

table.to_csv('resultats_simulation.csv', index=False)
table

,N,p,X_min=0,X_min=1,X_min=2,X_min=3,X_min=4,X_min=5,X_min=6,X_min=7,...,X_min=15,X_min=16,X_min=17,X_min=18,X_min=19,X_min=20,X_min=21,X_min=22,X_min=23,X_min=24
0,20,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9995,0.9983,0.9353,...,0.0067,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
1,20,6,1.0,0.9395,0.9395,0.9395,0.7365,0.4716,0.4716,0.3971,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
2,20,8,1.0,0.6626,0.6626,0.6626,0.4050,0.1315,0.1315,0.0975,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
3,25,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9782,...,0.1182,0.0316,0.0163,0.0100,0.0030,0.0001,0.0000,0.0000,0.0000,0.0
4,25,6,1.0,0.9861,0.9861,0.9861,0.9024,0.7749,0.7749,0.7032,...,0.0097,0.0013,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
5,25,8,1.0,0.8783,0.8783,0.8783,0.6743,0.4224,0.4224,0.3609,...,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
6,30,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9884,...,0.2669,0.1148,0.0842,0.0644,0.0221,0.0041,0.0027,0.0014,0.0001,0.0
7,30,6,1.0,0.9966,0.9966,0.9966,0.9566,0.8994,0.8994,0.8478,...,0.0578,0.0157,0.0088,0.0065,0.0023,0.0002,0.0001,0.0000,0.0000,0.0
8,30,8,1.0,0.9532,0.9532,0.9532,0.8239,0.6443,0.6443,0.5772,...,0.0066,0.0012,0.0003,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
